In [3]:
import re

Q. 1. Match a pair of words delimited by an equal sign and swap those words.

In [5]:
reobj = re.compile(r"(\w+)=(\w+)")
subject = "100=marks"
result = reobj.sub(r"\2=\1",subject)
result

# this will also work

# result  = re.sub(r"(\w+)=(\w+)",r"\2=\1|,subject) will also work


'marks=100'

Q.2. Re-attempt the above problem using named backreferences

In [4]:
reobj = re.compile(r"(?P<left>\w+)=(?P<right>\w+)")
subject = "100=marks"
result = reobj.sub(r"\g<right>=\g<left>",subject)
result


'marks=100'

Q.3 Suppose you have an HTML file in which various passages are marked as bold with <b> tags. Between each pair of bold tags, you want to replace all matches of the regular expression "before" with the replacement text "After".

Example: When processing the string : 

"before <b>first before</b> before <b>before before</b>", you want to end up with : 

"before <b>first after</b> before <b> after after</b>"

# Incorrect Solution

In [20]:
import re
html_string = "before <b>first before before</b> before <b>before before</b>"

pattern1 = r"(?s).*(?=.*\s<b>.*)(before).*(?=</b>.*)"

pattern2 = r"(?s).*?(?=.*?\s<b>.*?)(before).*?(?=</b>.*)"

print(re.findall(pattern1,html_string))

print(re.findall(pattern2,html_string))

['before']
['before', 'before']


What regex actually does ?  Regex does NOT enumerate “everything possible”. It performs: leftmost-first, non-overlapping search.

**Mechanism**

At any position:

1. Try to match the pattern

2. If success → record match

3. Move pointer to end of that match

4. Repeat

In [11]:
import re

text = "aaaa"
print(re.findall(r"aa", text))

['aa', 'aa']


The output is Not: ['aa', 'aa', 'aa']

Why: Because Matches are non-overlapping

**Overlapping requires explicit construction**

In [12]:
import re

text = "aaaa"
print(re.findall(r"(?=(aa))", text))

['aa', 'aa', 'aa']


**Backtracking misconception**

Regex does explore alternatives, but: only to find ONE valid match at a given position. It does NOT: return all possible matches at that position

# Case 1

In [ ]:
"""
(?s).*(?=.*\s<b>.*)(before).*(?=</b>.*)
"""

In [ ]:

"""
**Structure**

.*                     → greedy prefix - consume everything first.

(?=.*\s\<b>.*)          → assert: there exists " \<b>" somewhere ahead

(before)               → capture target

.*                     → greedy suffix

(?=\</b>.*)             → assert: there exists "\</b>" somewhere ahead

**Execution model**

**Step 1 — Greedy prefix**

.*  → consumes the entire string initially

Pointer moves to end.

**Step 2 — Backtracking**

Engine backtracks to allow:

(before) + (?=\</b>.*)

to succeed.

**Step 3 — First satisfiable configuration**

Engine finds the rightmost "before" such that:

there exists \</b> somewhere after it

**Result**

Only ONE match is produced

Captured value:

typically the last "before" inside the last \<b>...\</b>

"""

# Case 2

In [ ]:
"""
(?s).*?(?=.*?\s<b>.*)(before).*?(?=</b>.*)
"""

In [ ]:
"""
.*?                    → lazy prefix
(?=.*?\s<b>.*)         → assert: there exists " <b>" ahead
(before)               → capture
.*?                    → lazy suffix
(?=</b>.*)             → assert: there exists "</b>" ahead
"""

In [ ]:
"""
Execution model

Step 1 — Lazy prefix

.*? → start with smallest possible prefix (empty)

So engine tries matching at every position.

Step 2 — At each position

Try:

(before)

AND

(?=</b>.*)

Key condition

(?=</b>.*)

means:

there exists a closing </b> somewhere in the future

What matches

Any "before" satisfying:

∃ </b> somewhere later in the string

This includes:

- "before" outside <b>

- "before" inside <b>

Result

['before', 'before' ]

(all qualifying occurrences)

"""

In [ ]:
"""
What the problem actually requires

For a word "before" to be valid:

It must lie INSIDE a <b> ... </b> pair

Formally:

∃ "<b>" before the word

AND

∃ "</b>" after the word

AND

the nearest "</b>" after "<b>" must be after the word

This is a bounded interval constraint.

What your patterns actually enforce

Both patterns enforce only:

∃ "</b>" somewhere AFTER the word

and

∃ "<b>" somewhere in the string (via lookahead)

Why that is insufficient

Consider this word:

before <b>first before before</b> before <b>before before</b>

       ↑
       
       this "before"

This "before" is OUTSIDE <b>...</b>

But your condition:

(?=</b>.*)

is TRUE because:

there exists a </b> later in the string

So your regex accepts invalid cases.


"""

# Correct solution

In [5]:
import re

html_string = "before <b>first before before</b> before <b>before before</b>"

inner_re = re.compile("before")

def replace_within(matchobj):
    return inner_re.sub("after",matchobj.group())

result = re.sub("<b>.*?</b>",replace_within,html_string)

print(result)

before <b>first after after</b> before <b>after after</b>


In [6]:
""" 
Apply to your string
before <b>first before before</b> before <b>before before</b>

Iteration 1

Match:

<b>first before before</b>
Build result
append text before match:
"before "

append replacement:
"<b>first after after</b>"

So far:

before <b>first after after</b>

Pointer moves to end of first match.

Iteration 2

Next match:

<b>before before</b>
Append middle text
" before "
Append replacement
"<b>after after</b>"

Now:

before <b>first after after</b> before <b>after after</b>
End

No more matches → append remaining text (none here)

Key realization

No “joining” step exists

The string is constructed piece-by-piece during matching
Visual breakdown
[before ] 
+ [<b>first after after</b>] 
+ [ before ] 
+ [<b>after after</b>]
Why this works cleanly
Matches never overlap
→ each replacement occupies a distinct segment
→ assembly is deterministic
Final invariant
re.sub builds the output in a streaming manner:
    prefix + replacement + prefix + replacement + ...

"""

' \nApply to your string\nbefore <b>first before before</b> before <b>before before</b>\n\nIteration 1\n\nMatch:\n\n<b>first before before</b>\nBuild result\nappend text before match:\n"before "\n\nappend replacement:\n"<b>first after after</b>"\n\nSo far:\n\nbefore <b>first after after</b>\n\nPointer moves to end of first match.\n\nIteration 2\n\nNext match:\n\n<b>before before</b>\nAppend middle text\n" before "\nAppend replacement\n"<b>after after</b>"\n\nNow:\n\nbefore <b>first after after</b> before <b>after after</b>\nEnd\n\nNo more matches → append remaining text (none here)\n\nKey realization\n\nNo “joining” step exists\n\nThe string is constructed piece-by-piece during matching\nVisual breakdown\n[before ] \n+ [<b>first after after</b>] \n+ [ before ] \n+ [<b>after after</b>]\nWhy this works cleanly\nMatches never overlap\n→ each replacement occupies a distinct segment\n→ assembly is deterministic\nFinal invariant\nre.sub builds the output in a streaming manner:\n    prefix + 

Q.4. Positive lookahead — basic constraint

Match all words that are immediately followed by a digit

Example:

Input: abc1 def ghi2 xyz
Output: abc, ghi

In [7]:
import re
text = "abc1 def ghi24 xyz"
pattern1 = r"(\w+)(?=\d)" # ['abc', 'ghi2']
pattern2 = r"(?i)\b([a-z]+)\d+"

print(re.findall(pattern1, text))
print(re.findall(pattern2, text))

['abc', 'ghi2']
['abc', 'ghi']


Q5. Match all numbers NOT followed by "kg"

Example:

Input: 50kg 30m 100kg 25

Output: 30, 25

In [8]:
import re
text = "50kg 30m 100kg 25"
pattern1 = r"\d+(?!kg)" 
pattern2 = r"\d+(?!\d{0,}kg)" # \d{0,} equivalent to \d*

print(re.findall(pattern1, text))# Doesnt work!
print(re.findall(pattern2, text))# ['30', '25'] Works!

['5', '30', '10', '25']
['30', '25']


Q.6 Match all digits that are preceded by a "$"

Example:

Input: $100 €200 $50

Output: 100, 50

In [9]:
import re
text = "$100 €200 $50"
pattern1 = r"(?<=\$)(\d+)" 
# $ needs to be escaped as it is a metacharacter
print(re.findall(pattern1, text)) # ['100', '50']

['100', '50']


Q.7. Find all overlapping occurrences of "aba"

Example:

Input: ababa

Output: aba, aba

In [10]:
import re
text = "ababaa"
pattern1 = r"(?=aba)"
pattern2 = r"(?=(aba))"
print(re.findall(pattern1, text)) 
print(re.findall(pattern2, text)) 

['', '']
['aba', 'aba']


Q.8) Word extraction with internal constraint
Match words that contain at least one digit

Example:

Input: abc a1b xyz2 test

Output: a1b, xyz2

In [11]:
import re
text = "ababaa"
pattern = r"(?=(aba))"
print(re.findall(pattern, text))   

['aba', 'aba']


Q 9 .. The Password Complexity Riddle
Goal: Match a string that is 8–20 characters long and contains at least one uppercase letter, one lowercase letter, and one digit.

The Challenge: Solve this using only lookarounds and a single .* at the end. Why is the order of the lookarounds relevant (or is it)?

In [12]:
import re
text = "ABCabc123"
pattern = r"(?=[A-Z]).*?(?=[a-z]).*?(?=[0-9]).*"
print(re.findall(pattern, text))   

['ABCabc123']


In [13]:
""""
Why this pattern produces ['ABCabc123'] (in this specific case)
In this string, your characters are perfectly ordered: Uppercase first, then Lowercase, then Digits. Because the string is "pre-sorted," your pattern works by accident.

Here is the step-by-step cursor movement:

(?=[A-Z]): The cursor is at the start. It peeks, sees 'A' (Uppercase), and stays at the start.

.*?: The engine starts consuming characters one by one. It stops as soon as the next condition can be met. Since 'A' is already at the start, it actually consumes nothing here.

(?=[a-z]): It peeks. The next character is 'A'. This is not lowercase.

.*?: Now this "consumes" ABC until it reaches a position where the next character is lowercase. The cursor is now right before 'a'.

(?=[0-9]): It peeks. 'a' is not a digit.

.*?: This consumes abc until it reaches a position where the next character is a digit. The cursor is now right before '1'.

.*: This consumes everything left: 123.

The Fatal Flaw: The "Order" Trap
The reason this is not a good password validator is that it forces a specific order. If you change the input text, it fails.

Python
text = "123abcABC" # Same characters, different order
pattern = r"(?=[A-Z]).*?(?=[a-z]).*?(?=[0-9]).*"
# This will return an empty list [] """

'"\nWhy this pattern produces [\'ABCabc123\'] (in this specific case)\nIn this string, your characters are perfectly ordered: Uppercase first, then Lowercase, then Digits. Because the string is "pre-sorted," your pattern works by accident.\n\nHere is the step-by-step cursor movement:\n\n(?=[A-Z]): The cursor is at the start. It peeks, sees \'A\' (Uppercase), and stays at the start.\n\n.*?: The engine starts consuming characters one by one. It stops as soon as the next condition can be met. Since \'A\' is already at the start, it actually consumes nothing here.\n\n(?=[a-z]): It peeks. The next character is \'A\'. This is not lowercase.\n\n.*?: Now this "consumes" ABC until it reaches a position where the next character is lowercase. The cursor is now right before \'a\'.\n\n(?=[0-9]): It peeks. \'a\' is not a digit.\n\n.*?: This consumes abc until it reaches a position where the next character is a digit. The cursor is now right before \'1\'.\n\n.*: This consumes everything left: 123.\n\

In [14]:
import re
text = "ABCabc123"
pattern=r"^(?=.*[A-Z])(?=.*[a-z])(?=.*[0-9]).{8,20}$"
print(re.findall(pattern, text))  

['ABCabc123']


Q.10.  The Overlapping Match Problem
Goal: Count how many times the sequence aba occurs in abababa.

The Challenge: A standard regex like /aba/g will only find 2 matches because it consumes the characters. Write a regex using lookahead that allows you to find all 3 occurrences by matching the empty string at the start of each occurrence.

In [22]:
import re 
pattern = r'(?=aba)'
text="abababa"
print(len(re.findall(pattern,text)))

3


In [ ]:
import re
text = "abababa"
pattern=r"(?=aba)\w"
print(re.findall(pattern, text))

['a', 'a', 'a']


Step 1: Index 0 (Before the first 'a')

Lookahead (?=aba): The scout looks ahead and sees aba. Match found. The scout returns to index 0.

Consuming \w: The main cursor moves forward and "eats" the first letter: a.

Current Result: ['a']

Remaining String: bababa

Step 2: Index 1 (Before the first 'b')

Lookahead (?=aba): The scout looks ahead. It sees bab. That is not aba. Match fails.

The engine moves the cursor to the next position.

Remaining String: ababa

Step 3: Index 2 (Before the second 'a')

Lookahead (?=aba): The scout looks ahead and sees aba (the middle part of your string). Match found. The scout returns to index 2.

Consuming \w: The main cursor moves forward and "eats" the letter: a.

Current Result: ['a', 'a']

Remaining String: baba

Step 4: Index 3

Lookahead (?=aba): Sees bab. Match fails.

Engine moves the cursor.

Step 5: Index 4 (Before the third 'a')

Lookahead (?=aba): Sees aba (the final three letters). Match found.

Consuming \w: The cursor eats the letter: a.

Current Result: ['a', 'a', 'a']


In [20]:
import re
text = "abababa"
pattern=r"(?=aba)"
print(re.findall(pattern, text))  

['', '', '']



Without Consumption ((?=aba))
Engine checks lookahead at Index 0 (Success).

Nothing consumes anything. Cursor is still at Index 0.

The engine forces the cursor to Index 1 to prevent an infinite loop.

What if it didn't move?
If the engine didn't have this "forced nudge," your code re.findall(r"(?=aba)", "abababa") would never finish running. It would just keep finding the first empty match at the very beginning of the string until your computer ran out of memory or you killed the process.

This is why findall returns multiple results even for zero-width patterns—it's essentially saying: "I found a match here, now I'm going to step forward one inch and see if I can find another one."

Q.11. Negative Lookbehind with Variable Length
Goal: Match the word "cat" only if it is not preceded by the word "wild" or "mountain".

The Challenge: Many regex engines (like JavaScript or Python’s re module) traditionally do not support variable-length lookbehinds. How would you write this for an engine that does support it (like .NET or PCRE), and how would you work around it in an engine that doesn't?

In [1]:
import re
pattern = r'(?<!wild)(?<!mountain)(cat)'
text = "wildcatmountaincatdogcat"
print(re.findall(pattern, text)) 

['cat']


# Incorrect solution

In [2]:
import re
pattern = r'(?<!wild)(cat)|(?<!mountain)(cat)'
text = "wildcatmountaincatdogcat"
print(re.findall(pattern, text)) 

[('', 'cat'), ('cat', ''), ('cat', '')]


In [ ]:
"""
Here is the step-by-step mechanical process:

1. The Global Search
The engine scans the string from left to right. It isn't looking for the lookbehinds yet; it is looking for the first literal "cat".

2. Encountering "wildcat" (at index 4)
The engine finds the characters c-a-t. Now it checks its "branches" to see if this "cat" is valid.

Attempting Branch 1 ((?<!wild)(cat)):

It looks behind the 'c'. It sees "wild".

Because it’s a negative lookbehind, it says: "I cannot match if 'wild' is there."

Branch 1 Fails.

Attempting Branch 2 ((?<!mountain)(cat)):

It looks behind the 'c'. It sees "wild".

It asks: "Is this 'mountain'?" No, it's "wild".

Since it is not "mountain", the condition is satisfied.

Branch 2 Succeeds.

Result: The engine records a match for the second capturing group.

3. Encountering "mountaincat" (at index 15)
The engine moves forward and finds the next c-a-t.

Attempting Branch 1 ((?<!wild)(cat)):

It looks behind the 'c'. It sees "tain" (the end of "mountain").

It asks: "Is this 'wild'?" No.

The condition is satisfied.

Branch 1 Succeeds.

Attempting Branch 2:

The engine usually stops as soon as one branch of an alternation (|) succeeds.

Result: The engine records a match for the first capturing group.

4. Encountering "dogcat" (at index 21)
The engine finds the third c-a-t.

Attempting Branch 1:

It looks behind. It sees "dog".

Is it "wild"? No.

Branch 1 Succeeds.

Result: Match recorded for the first group.

"""

Q.11  The "Not-Followed-By" Chain
Goal: Match a sequence of digits that is not followed by "USD" and not followed by "EUR".

The Challenge: Can you achieve this using two separate negative lookahead assertions? Does it matter if they are nested or sequential?

Example: \d+(?!USD)(?!EUR)

In [ ]:
import re
text = "100 USD 50 EUR 200 INR 250 EUR 100 USD 200INR"

# Use (?!\d) to ensure we've captured ALL digits of the number 
# before we check if it's followed by USD or EUR.
pattern = r'(\d+)(?!\d)(?!(?:\s)?USD|(?:\s)?EUR)'

print(re.findall(pattern, text)) 
# Output: ['200', '200']

['200', '200']


: 

In [22]:
import re
pattern = r'(\d+(\s)?)(?!USD|EUR)'
text = "100 USD 50 EUR 200 INR 250 EUR 100 USD 200INR"
print(re.findall(pattern, text)) 

[('100', ''), ('50', ''), ('200 ', ' '), ('250', ''), ('100', ''), ('200', '')]


In [13]:
import re
pattern = r'(\d+)(?!(\s)?USD|(\s)?EUR)'
text = "100 USD 50 EUR 200 INR 250 EUR 100 USD 200INR"
print(re.findall(pattern, text)) 

[('10', '', ''), ('5', '', ''), ('200', '', ''), ('25', '', ''), ('10', '', ''), ('200', '', '')]


In [ ]:
""""
1. The "Greedy" Digits vs. The Lookahead
This is a classic regex trap. Let's look at what happens when 
the engine sees 100 USD:

(\d+): The engine is "greedy." It sees "100" and grabs all 
three digits.

The Lookahead: It peeks ahead to see if the next part is 
[space]USD. It is [space]USD.

The Failure?: You would expect it to fail here. But, the 
engine is smart. 
It realizes that if it only matches "10" (two digits) instead 
of "100", the next character is "0".

The Loophole: Since "0" is not [space]USD, the lookahead 
passes!

2. Why the output is a tuple of 3 items
Because you have three sets of parentheses, re.findall returns a tuple for every match:

Group 1: (\d+) — The number (e.g., '100').

Group 2: (\s)? — The optional space before USD (returns '' because it didn't match).

Group 3: (\s)? — The optional space before EUR (returns '' because it didn't matc
"""

In [ ]:
"""
what is the difference between r'(\d+(\s)?)(?!USD|EUR)' and
r'(\d+(\s)?)(?!USD)|(?!EUR)'

This is a fantastic question that touches on the "invisible" 
logic of how regex branches (the | operator) actually behave.

The difference lies in Exclusive Logic vs. Permissive Logic. 
In short: the first pattern is a strict filter, while the 
second pattern is essentially "broken" and will match almost 
everything.

1. r'(\d+(\s)?)(?!USD|EUR)' (The Strict Filter)
This uses a single lookahead with an internal OR.

Logic: "Find digits, then check the next part. If it is USD OR
if it is EUR, reject the match."

Result: It successfully blocks both 100 USD and 50 EUR. It 
only wants things that are neither.

2. r'(\d+(\s)?)(?!USD)|(?!EUR)' (The "Always True" Trap)
This splits the lookaheads into two separate branches. This 
is almost always a mistake in regex logic.

Logic: "Find digits. Now, check if the next part is NOT USD...
OR... check if the next part is NOT EUR."

The Problem: If the text is 100 USD, the first branch (?!USD)
fails. But the engine then tries the second branch (?!EUR). 
Since " USD" is definitely not "EUR", the second branch returns
True.

Result: Every single number in your string will match. Even 
100 USD matches because "USD" satisfies the "is not EUR" 
condition.


"""

"""
Q12. Preventing "The Greedy Trap"
Goal: You have a string of HTML tags. You want to match the content inside <div> tags, but only if the <div> does not contain the word "Error".

ex: 
text = ""
<div>System Status: OK</div>
<div>System Status: Error - Failed to connect</div>
<div>User Profile: Loaded</div>
""

# Output: ['<div>System Status: OK</div>', '<div>User Profile: Loaded</div>']

The Challenge: Explain why <div>(?!.*Error).*?</div> might still match a <div> containing "Error" if there is another <div> further down the string. How do you "anchor" the negative lookahead to every character inside the tag?

"""

In [ ]:
import re
pattern = r'(\d+)(?!(\s)?USD|(\s)?EUR)'
text = "100 USD 50 EUR 200 INR 250 EUR 100 USD 200INR"
print(re.findall(pattern, text)) 



6. Atomic Lookarounds and Performance
Goal: Match a long string of 'a's followed by a 'b', but fail fast if 'b' is missing.

The Challenge: Compare the performance of (a+)+b versus a version using lookahead to simulate an atomic group: (?=(a+))\1b. Explain how the lookahead prevents catastrophic backtracking when the match fails.

7. The Comma-Separator Logic
Goal: Match every digit in a large number (e.g., 1234567) that should be followed by a comma according to standard US grouping (thousands).

The Challenge: Use a positive lookahead with a quantifier and an anchor to identify these positions.

Target: In 1234567, it should match 1, 4.

8. The "Middle-Only" Exclusion
Goal: Match any word that contains the letter "e", but the "e" cannot be the first or the last letter of the word.

The Challenge: Use a combination of lookahead and lookbehind to restrict the "e" based on its surrounding context within \b\w+\b.

9. Nested Lookarounds
Goal: Explain what this regex does: (?!(?<=A)B).

The Challenge: This is a negative lookahead containing a positive lookbehind. Under what specific condition will this match the position between characters in the string AB?

10. Floating Point Validation
Goal: Match a number only if it contains a decimal point, but ensure the decimal point is not the last character in the string.

The Challenge: Use a positive lookahead to confirm the existence of the dot somewhere in the string, while using a negative lookahead to ensure the string doesn't end with it. Write this without using "or" (|) logic.

) Exclude content inside tags (core difficulty)
Match all double quotes (") that are OUTSIDE HTML tags

In [16]:
"""
Example:

Input: "text" <span class="a">"inner"</span>

Output: the two quotes around text only
"""

'\nExample:\n\nInput: "text" <span class="a">"inner"</span>\n\nOutput: the two quotes around text only\n'

# Incorrect Solution

In [ ]:
import re
text = "\"text\" <span class=\"a\">\"inner\"</span>"
pattern = r"(\w+)(?!<[^>]+>)|(?!<[^>]+>)(\w+)"
print(re.findall(pattern, text))   

[('text', ''), ('span', ''), ('class', ''), ('a', ''), ('inner', ''), ('span', '')]


In [ ]:
"""
What it actually does

Both branches reduce to:

\w+  with a weak condition about tags ahead
Branch 1
(\w+)(?!<[^>]+>)

→ match a word not immediately followed by <...>

Branch 2
(?!<[^>]+>)(\w+)

→ match a word not starting at a position where a tag begins

Why this is incorrect

Your requirement:

match text OUTSIDE tags

Your pattern checks:

“is a tag immediately ahead?”

This is a local check, not a region constraint.

Concrete failure
"text" <span class="a">"inner"</span>

Your pattern will match:

text   ✔ (correct)
inner  ✘ (incorrect — inside tag content)
span   ✘ (inside tag)
class  ✘ (inside tag)
a      ✘ (inside tag)
Core issue
(?!<[^>]+>)

means:

“the next characters do not form a tag”

It does NOT mean:

“I am outside a tag region”
"""

In [ ]:
"""
Why is span getting matched?

Evaluate "span"

Text:

"text" <span class="a">"inner"</span>
         ^

At the s in "span":

Branch 1
(\w+)(?!<[^>]+>)
\w+ → matches "span"
Now check (?!<[^>]+>):
what comes immediately after "span"?
→ space " "

So:

<[^>]+> does NOT start here → condition is TRUE

✔ Branch 1 matches "span"
"""

In [ ]:
import re
text = "\"text \" <span class=\"a\">\"inner\"</span>"
pattern = r'(\w+)(?!<.*+>.*+<.*+>)(\w+)'
print(re.findall(pattern, text))

[('tex', 't'), ('spa', 'n'), ('clas', 's'), ('inne', 'r'), ('spa', 'n')]


In [ ]:
"""
Example: "text"

Initial greedy attempt:

(\w+) → "text"
(\w+) → fails (nothing left)

So engine backtracks:

(\w+) → "tex"
(\w+) → "t"

Your pattern (relevant part)
(\w+)   (?!...)   (\w+)

Execution model:

1. (\w+) consumes characters

2. (?!...) checks a condition at the CURRENT 
position. (?!...) does NOT consume characters
So the second \w+ does not match “after” anything. It matches at the same position where the first group stopped.

3. (\w+) continues from that SAME position

"""

In [25]:
import re
text = "\"text\" <span class=\"a\">\"inner\"</span>"
pattern = r'(\w+?)(\b)(?:<.*?>.*?<.*?>)(\b)(\w+)?'
print(re.findall(pattern, text))

[]



9) Conditional structure via lookahead
Match words only if they are followed somewhere later in the string by "END"

Example:

Input: start middle END ignore
Output: start, middle
10) Multi-condition constraint (AND logic)
Match words that:
    contain at least one digit
    AND end with a letter

Example:

Input: a1b 123a ab1 1abc xyz
Output: a1b, 123a

Q.4. Suppose you have an HTML file in which you want to replace straight double quotes with single quotes but only the quotes outside of HTML tags. Quotes within HTML tags must remain the same. 


In [24]:
"""
"text" <span class="middle">"text"</span> 

must turn into 

'text' <span class="middle">'text'</span>
"""

'\n"text" <span class="middle">"text"</span> \n\nmust turn into \n\n\'text\' <span class="middle">\'text\'</span>\n'

In [25]:
import re
html_file = "\"text\" <span class=\"middle\">\"text\"</span>"
pattern = r"(?!<.*>)"
replace_what = re.compile("\"")

def func(match_obj):
    return replace_what.sub("'",match_obj)


result = re.sub(pattern,func,html_file)

print(result)


TypeError: expected string or bytes-like object, got 're.Match'